# Stage 2: Generate Conversations

This notebook generates Stage 2 training data using GPT-4.1.

For each TEM subfigure, GPT receives the Visual Audit and two descriptions generated in Stage 1, and converts them into two separate multi-turn conversations:

- **VisionGround** — conversation based solely on visually observable features
- **DomainContext** — conversation incorporating domain-specific terms grounded in visible features

# Import packages

In [1]:
import os
import gc
import pandas as pd
import csv
from pathlib import Path
from openai import OpenAI

## Settings

Adjust the paths below to match your local setup:

- `INPUT_CSV_PATH`: CSV output from `stage1_generate_descriptions.ipynb`, containing `CROP_IMAGE`, `reasoning_process`, `visionground_description`, and `domaincontext_description` columns
- `OUTPUT_CSV_PATH`: output CSV where generated conversations will be saved

The OpenAI API key is read from the `OPENAI_API_KEY` environment variable.

In [ ]:
INPUT_CSV_PATH  = "/path/to/your/stage1_descriptions.csv"
OUTPUT_CSV_PATH = "/path/to/your/stage2_conversations.csv"

API_Key = os.getenv("OPENAI_API_KEY")
client  = OpenAI(api_key=API_Key)

## Define Prompts

In [ ]:
def system_prompt():
    return """
You are a helpful AI assistant designed to generate multi-turn conversations
about scientific microscopy images for multimodal model training.

You do NOT have access to the image itself.

Instead, you are given a Visual Audit and two kinds of textual descriptions

1) VisionGround Description:
   Descriptions derived solely from verified visual evidence in the image.

2) DomainContext Description:
   Descriptions that incorporate caption-derived information together with
   visual evidence from the image.
   
IMPORTANT RULES:
- Treat the Visual Audit as the single source of truth about the image.
- Do NOT add, infer, or speculate beyond the provided information.
- Do NOT refer to captions, audits, or metadata in the conversation.

Your task is to generate TWO SEPARATE multi-turn conversations
based on the same image by transforming the given descriptions
into natural question–answer exchanges.

--------------------------------
Conversation A: VisionGround
--------------------------------
- Transform the VisionGround Description into a multi-turn conversation.
- Use only generic visual language.

--------------------------------
Conversation B: DomainContext
--------------------------------
- Transform the DomainContext Description into a multi-turn conversation.
- Domain-specific terms may be used only if they are grounded in visible features.

For BOTH conversations:
- The conversation should resemble a natural question–answer exchange,
  similar to LLaVA training conversations.
- Each conversation should contain as many turns as possible,
  while remaining fully grounded in the provided descriptions
  and without introducing new or speculative information.
- The assistant’s answers must remain factual, and image-grounded.
- Do NOT mix the two modes.
- Do NOT mention the audit or caption explicitly.

OUTPUT FORMAT (JSON only):
{
  "VisionGround_conversations": [
    {
      "from": "human",
      "value": "<image>\\n ..."
    },
    {
      "from": "gpt",
      "value": "..."
    }
  ],
  "DomainContext_conversations": [
    {
      "from": "human",
      "value": "<image>\\n ..."
    },
    {
      "from": "gpt",
      "value": "..."
    }
  ]
}
"""

In [ ]:
def build_user_prompt(visual_audit, visionground_text, domaincontext_text):
    return f"""
[Visual Audit]
{visual_audit}

[VisionGround Description]
{visionground_text}

[DomainContext Description]
{domaincontext_text}
"""

## Check Progress

Loads already-processed images from `OUTPUT_CSV_PATH` so the notebook can be safely resumed after interruption.

In [ ]:
done_images = set()
if os.path.exists(OUTPUT_CSV_PATH):
    with open(OUTPUT_CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if row:
                done_images.add(row[0])
print(f"{len(done_images)} images already processed")

## Generate Conversations

For each subfigure in `INPUT_CSV_PATH`:
1. Skips images already processed
2. Sends the Visual Audit and two descriptions to GPT-4.1
3. Saves the generated VisionGround and DomainContext conversations incrementally to `OUTPUT_CSV_PATH`

In [ ]:
df = pd.read_csv(INPUT_CSV_PATH, dtype=str, encoding="utf-8")

file_exists = os.path.exists(OUTPUT_CSV_PATH)

with open(OUTPUT_CSV_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["CROP_IMAGE", "RESPONSE"])
    
    for idx, row in df.iterrows():
        image_name = row['CROP_IMAGE']

        if image_name in done_images:
            continue
    
        print(f"Processing {idx}: {image_name}")

        visual_audit      = row['reasoning_process']
        visionground_text = row['visionground_description']
        domaincontext_text = row['domaincontext_description']

        try:
            response = client.responses.create(
                model="gpt-4.1",
                temperature=0.25,
                input=[
                    {
                        "role": "system",
                        "content": [{"type": "input_text", "text": system_prompt()}],
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "input_text", "text": build_user_prompt(visual_audit, visionground_text, domaincontext_text)},
                        ],
                    },
                ],
            )

            output = response.output_text
            writer.writerow([row['CROP_IMAGE'],output])
            f.flush()
            os.fsync(f.fileno())

            done_images.add(image_name)
            print(f"Save: {image_name}")
        except Exception as e:
            print(f"Error on {image_name}: {e}")
            continue
            
        del response
        gc.collect()